In [1]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
from typing import List, Tuple, Dict, Any, Optional

import matplotlib.pyplot as plt

In [ ]:
# Prefer compressed COCO RLE if available
try:
    from pycocotools import mask as mask_utils
    HAS_COCO = True
except Exception:
    HAS_COCO = False

### helper Functions
def find_pairs(src: Path, label_suffix="_instance") -> list[tuple[Path, Path]]:
    """Return (image_path, label_path) for files like path_{05.d}.png and path_{05.d}_instance.png."""
    pairs = []
    for img_path in sorted(src.glob("*.png")):
        name = img_path.name
        if name.endswith(f"{label_suffix}.png"):
            # skip label files in the image scan
            continue
        base = name[:-4]  # strip ".png"
        lbl_path = src / f"{base}{label_suffix}.png"
        if lbl_path.exists():
            pairs.append((img_path, lbl_path))
    return pairs

def load_png(p: Path) -> np.ndarray:
    arr = np.array(Image.open(p))
    return arr

def bbox_from_mask(mask: np.ndarray) -> List[int]:
    ys, xs = np.where(mask)
    if ys.size == 0: return [0, 0, 0, 0]
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    return [x0, y0, x1 - x0, y1 - y0]


print(HAS_COCO)


True


In [10]:
src = "/Volumes/warm_SD/data/160/00_series/00/160_00_01_50x_patches"
out = "../data/ds1"
label_suffix = "_instance"

src = Path(src)
out = Path(out)
(out / "images").mkdir(parents=True, exist_ok=True)
(out / "annotations").mkdir(parents=True, exist_ok=True)


min_area = 1
default_uncertain_iou = 1.0
category_id = 1

pairs = find_pairs(src, label_suffix=label_suffix)
if not pairs:
    print("No matching pairs found.")


for n, (img_path, lbl_path) in enumerate(pairs):
    stem = img_path.stem  # includes braces/dots verbatim, e.g., "path_{05.d}"
    # load
    img = load_png(img_path)        # [H,W] uint16 grayscale
    lbl = load_png(lbl_path).astype(np.uint16)        # [H,W] uint16 label IDs

    if img.shape != lbl.shape:
            print(f"Skip (shape mismatch): {img_path.name} vs {lbl_path.name}")
            continue
    
    img_rgb = np.dstack([img, img, img])


    # save
    W, H, _ = img_rgb.shape
    out_img = out  / "images" / f"{stem}.jpg"
    out_json = out  / "annotations" / f"{stem}.json"


    Image.fromarray(img_rgb, mode="RGB").save(out_img, format="JPEG")

    # Build per-image annotations list
    ann_list = []
    ids = np.unique(lbl)
    ids = ids[ids != 0]  # 0 = background
    for inst_id in ids:
        mask = (lbl == int(inst_id))
        area = int(mask.sum())

        if area < min_area:
            continue
        # make Fortran-contiguous uint8 mask
        m = np.asfortranarray(mask.astype(np.uint8))
        rle = mask_utils.encode(m)
        rle["counts"] = rle["counts"].decode("ascii")  # <-- critical fix

        ann_list.append({
            "area": area,
            "bbox": bbox_from_mask(mask),
            "iscrowd": 0,
            "category_id": category_id,
            "instance_id": int(inst_id),
            "uncertain_iou": float(default_uncertain_iou),
            "segmentation": {"size": [int(mask.shape[0]), int(mask.shape[1])],
                            "counts": rle["counts"]}
        })


    # Write ONE JSON per image with top-level "annotations"

    with open(out_json, "w") as f:
        json.dump({"annotations": ann_list}, f, indent=2)






    # if n == 2:
    #      break


